# Ir Além — Transfer Learning, Fine Tuning e segmentação

**Entrega opcional da Fase 6** (não vale nota; concorre ao troféu do "Ir Além"). Este notebook parte da mesma base da Entrega 1 e testa **duas hipóteses**:

1. **Transfer Learning**: uma rede grande pré-treinada na ImageNet (*MobileNetV2*) performa melhor que uma arquitetura treinada do zero (a CNN da Entrega 2) quando temos apenas 64 imagens de treino?
2. **Pré-segmentação**: entregar à rede a informação de *quais pixels importam* — recortando o objeto com uma máscara e apagando o fundo — facilita a classificação?

**Roteiro:** segmentar as imagens automaticamente (YOLO pré-treinado no COCO, que conhece `cow` e `truck`) → aplicar a máscara recortando o fundo → treinar os dois classificadores nas duas versões da base (original e recortada) → comparar acurácia, tempo de treino e tempo de inferência.

> Para rodar: *Ambiente de execução → Alterar tipo de ambiente → **T4 GPU*** e **Reiniciar e executar tudo**. O notebook baixa a base sozinho e leva cerca de 10 minutos.

## 1. Ambiente e base de imagens

Mesmo dataset da Entrega 1 (pasta pública do Drive do grupo) e o modelo de segmentação pré-treinado no COCO, que já conhece as duas classes do projeto (`cow` = 19 e `truck` = 7).

In [ ]:
# Ambiente: dataset da Entrega 1 + modelo de segmentação do COCO
!pip install -q ultralytics

from google.colab import drive
drive.mount('/content/drive')

import glob, os, shutil
import pandas as pd

DRIVE_DATASET_ID = '1sd36wCw_DFbHvUJQ5rCEoY0kv68eD0WU'  # pasta "dataset" do grupo no Drive (pública)
LOCAL = '/content/dataset'                                # base original
PASTA_OBJ = '/content/dataset_objeto'                     # base com o fundo recortado

# 1) copia a pasta montada, se existir no Drive montado
CANDIDATOS = [
    '/content/drive/MyDrive/Espaço Matopiba/Fase 6 - Cap 1/dataset',
    '/content/drive/MyDrive/Fase 6 - Cap 1/dataset',
    '/content/drive/MyDrive/dataset',
]
for raiz in ('/content/drive/MyDrive', '/content/drive/Shareddrives'):
    CANDIDATOS += glob.glob(f'{raiz}/*/dataset') + glob.glob(f'{raiz}/*/*/dataset')
DRIVE_DATASET = next((p for p in CANDIDATOS if os.path.isdir(f'{p}/images/train')), None)

if os.path.exists(LOCAL):
    shutil.rmtree(LOCAL)
if DRIVE_DATASET:
    shutil.copytree(DRIVE_DATASET, LOCAL)
    print('dataset copiado da pasta montada:', DRIVE_DATASET)
else:
    # 2) pasta apenas compartilhada não aparece no mount: baixa pela API do Drive
    #    (o gdown não serve: o Google limita a 50 arquivos por pasta e a nossa tem 164)
    from google.colab import auth
    from googleapiclient.discovery import build
    from googleapiclient.http import MediaIoBaseDownload
    auth.authenticate_user()
    svc = build('drive', 'v3')

    def filhos(fid):
        return svc.files().list(q=f"'{fid}' in parents and trashed = false", pageSize=1000,
                                fields='files(id,name,mimeType)', supportsAllDrives=True,
                                includeItemsFromAllDrives=True).execute()['files']

    pasta = next(f for f in filhos(DRIVE_DATASET_ID)
                 if f['name'] == 'dataset' and f['mimeType'] == 'application/vnd.google-apps.folder')

    def baixa(fid, destino):
        for f in filhos(fid):
            alvo = f'{destino}/{f["name"]}'
            if f['mimeType'] == 'application/vnd.google-apps.folder':
                os.makedirs(alvo, exist_ok=True)
                baixa(f['id'], alvo)
            else:
                os.makedirs(destino, exist_ok=True)
                with open(alvo, 'wb') as arq:
                    dl = MediaIoBaseDownload(arq, svc.files().get_media(fileId=f['id'], supportsAllDrives=True))
                    pronto = False
                    while not pronto:
                        _, pronto = dl.next_chunk()
    baixa(pasta['id'], LOCAL)
    print('dataset baixado pela API do Drive')

# índice das imagens: a classe vem do prefixo do nome (vaca_* / caminhao_*), como na Entrega 2
def indice(split, raiz=LOCAL):
    linhas = [{'filename': p, 'classe': 'vaca' if os.path.basename(p).startswith('vaca') else 'caminhao'}
              for p in sorted(glob.glob(f'{raiz}/images/{split}/*.jpg'))]
    return pd.DataFrame(linhas)

for split in ('train', 'val', 'test'):
    d = indice(split)
    print(split, len(d), 'imagens |', d['classe'].value_counts().to_dict())

## 2. Segmentação automática e recorte do objeto

A máscara vem de um **YOLO de segmentação pré-treinado no COCO** (`yolov8s-seg.pt`), rodando **sem treino nosso**: pedimos só as classes que correspondem aos nossos objetos (`truck` = 7 e `cow` = 19). Para cada imagem usamos a máscara da **classe certa** (a que está no nome do arquivo) e unimos as máscaras de todos os objetos daquela classe — em `vaca_013`, por exemplo, são vários animais.

A imagem recortada mantém só a região mascarada e pinta o resto de **preto**, que é o pré-processamento extra pedido no enunciado.

In [ ]:
# Segmentação: devolve a máscara da classe pedida, no tamanho da imagem original
import numpy as np
from PIL import Image
from ultralytics import YOLO

seg = YOLO('yolov8s-seg.pt')            # pesos do COCO, baixados na primeira execução
COCO_ID = {'vaca': 19, 'caminhao': 7}   # nossa classe -> índice no COCO

def mascara_do_objeto(caminho, classe, conf=0.25):
    """Roda a segmentação e devolve (máscara booleana, tamanho original) da classe pedida; None se não achou."""
    r = seg.predict(caminho, classes=[COCO_ID[classe]], conf=conf, verbose=False)[0]
    if r.masks is None:
        return None
    h, w = r.orig_shape
    mascara = np.zeros((h, w), dtype=bool)
    for m in r.masks.data.cpu().numpy():          # (n, h_rede, w_rede) com 0/1
        pequena = Image.fromarray((m * 255).astype('uint8')).resize((w, h))
        mascara |= np.array(pequena) > 127        # união dos objetos da classe
    return mascara

def recorta(caminho, mascara, cor_fundo=0):
    """Aplica a máscara: mantém o objeto e pinta o fundo de preto."""
    img = np.array(Image.open(caminho).convert('RGB'))
    saida = np.full_like(img, cor_fundo)
    saida[mascara] = img[mascara]
    return saida

# Gera a base recortada (mesmos nomes, mesmo split) e guarda um exemplo de cada classe
if os.path.exists(PASTA_OBJ):
    shutil.rmtree(PASTA_OBJ)

exemplos, sem_mascara, cobertura = [], [], []
for split in ('train', 'val', 'test'):
    for p in sorted(glob.glob(f'{LOCAL}/images/{split}/*.jpg')):
        nome = os.path.basename(p)
        classe = 'vaca' if nome.startswith('vaca') else 'caminhao'
        m = mascara_do_objeto(p, classe)
        destino_dir = f'{PASTA_OBJ}/images/{split}'
        os.makedirs(destino_dir, exist_ok=True)
        if m is None:                             # não achou o objeto: entra com a imagem original, e fica registrado
            sem_mascara.append(f'{split}/{nome}')
            imagem = np.array(Image.open(p).convert('RGB'))
        else:
            cobertura.append(m.mean())
            imagem = recorta(p, m)
            if len(exemplos) < 8 and (classe, split) not in {(c, s) for c, s, _, _ in exemplos}:
                exemplos.append((classe, split, p, m))
        Image.fromarray(imagem).save(f'{destino_dir}/{nome}')

print(f'imagens sem máscara: {len(sem_mascara)} de 80 {sem_mascara}')
print(f'cobertura média da máscara: {np.mean(cobertura):.1%} da área da imagem')
print('base recortada em', PASTA_OBJ)

## 3. Original, máscara e imagem recortada (evidência do pré-processamento)

Para algumas imagens da base, na ordem pedida pelo enunciado: **imagem original**, **máscara obtida** pela segmentação e **imagem com o fundo recortado**.

In [ ]:
# Figura com original | máscara | recortada, para 2 imagens de cada classe
import matplotlib.pyplot as plt

fig, eixos = plt.subplots(4, 3, figsize=(9, 12))
for linha, (classe, split, caminho, mascara) in enumerate(exemplos[:4]):
    original = np.array(Image.open(caminho).convert('RGB'))
    eixos[linha, 0].imshow(original); eixos[linha, 0].set_title(f'original — {os.path.basename(caminho)}', fontsize=9)
    eixos[linha, 1].imshow(mascara, cmap='gray'); eixos[linha, 1].set_title('máscara da segmentação', fontsize=9)
    eixos[linha, 2].imshow(recorta(caminho, mascara)); eixos[linha, 2].set_title('com o fundo recortado', fontsize=9)
    for e in eixos[linha]:
        e.axis('off')
plt.tight_layout()
plt.savefig('/content/segmentacao_exemplos.png', dpi=110)
plt.show()
print('figura salva em /content/segmentacao_exemplos.png')

## 4. Os dois classificadores

Mesmo esquema da Entrega 2: entrada 224×224, augmentação no treino (rotação, flip e zoom), split 32/4/4 por classe e saída sigmoide.

- **CNN do zero**: a mesma arquitetura da seção 9 do notebook principal (3 blocos `Conv2D + MaxPooling` + densas).
- **MobileNetV2 (Transfer Learning + Fine Tuning)**: pesos da ImageNet, `include_top=False`, cabeça nova com `GlobalAveragePooling2D` + `Dropout` + uma saída sigmoide. **Duas fases**: primeiro treina só a cabeça com a base congelada; depois descongela as **últimas 30 camadas** (`base.layers[:-30]`) com learning rate baixa (1e-5).

**Por que MobileNetV2?** É leve (≈2,2 M de parâmetros com `include_top=False`), treina rápido em 64 imagens e é a menor das redes sugeridas pelo enunciado — o que importa aqui é testar a hipótese, não ganhar precisão bruta. **Por que congelar tudo na primeira fase?** Com poucas imagens, atualizar milhões de parâmetros de uma vez destrói os filtros aprendidos na ImageNet (catastrophic forgetting); treinar a cabeça primeiro dá um ponto de partida estável. **Por que descongelar só as últimas camadas?** As primeiras camadas guardam bordas e texturas genéricas; as últimas guardam conceitos específicos — são essas que queremos ajustar (fine tuning), e com LR baixo para não desfazer o que já funciona.

In [ ]:
# Infra comum: índices, geradores com augmentação, treino e avaliação
import time
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

IMG, LOTE = 224, 8
tf.random.set_seed(0)

def geradores(raiz):
    """Geradores para a base em `raiz`: augmentação no treino; validação e teste apenas normalizados."""
    treino = ImageDataGenerator(rescale=1/255, rotation_range=20, horizontal_flip=True, zoom_range=0.2)
    simples = ImageDataGenerator(rescale=1/255)
    return (treino.flow_from_dataframe(indice('train', raiz), x_col='filename', y_col='classe',
                                       target_size=(IMG, IMG), batch_size=LOTE, class_mode='binary', seed=0),
            simples.flow_from_dataframe(indice('val', raiz), x_col='filename', y_col='classe',
                                        target_size=(IMG, IMG), batch_size=LOTE, class_mode='binary', shuffle=False),
            simples.flow_from_dataframe(indice('test', raiz), x_col='filename', y_col='classe',
                                        target_size=(IMG, IMG), batch_size=LOTE, class_mode='binary', shuffle=False))

def cnn_do_zero():
    tf.random.set_seed(0)
    modelo = models.Sequential([
        layers.Input((IMG, IMG, 3)),
        layers.Conv2D(32, 3, activation='relu'), layers.MaxPooling2D(),
        layers.Conv2D(64, 3, activation='relu'), layers.MaxPooling2D(),
        layers.Conv2D(128, 3, activation='relu'), layers.MaxPooling2D(),
        layers.Flatten(),
        layers.Dense(128, activation='relu'), layers.Dropout(0.3),
        layers.Dense(1, activation='sigmoid'),
    ])
    modelo.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return modelo, None

def mobilenet():
    tf.random.set_seed(0)
    base = MobileNetV2(input_shape=(IMG, IMG, 3), include_top=False, weights='imagenet')
    base.trainable = False                       # fase 1: base congelada
    modelo = models.Sequential([base, layers.GlobalAveragePooling2D(), layers.Dropout(0.2),
                                layers.Dense(1, activation='sigmoid')])
    modelo.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='binary_crossentropy', metrics=['accuracy'])
    return modelo, base

def treina(construtor, ger_treino, ger_val, epocas=40):
    """Treina o modelo e devolve (modelo, históricos das fases, tempo em minutos)."""
    modelo, base = construtor()

    def callback():   # um EarlyStopping novo por fase (o callback guarda estado entre fits)
        return tf.keras.callbacks.EarlyStopping(patience=8, restore_best_weights=True, monitor='val_loss')

    t0 = time.time()
    h1 = modelo.fit(ger_treino, validation_data=ger_val, epochs=epocas, callbacks=[callback()])
    h2 = None
    if base is not None:                          # fase 2: fine tuning das últimas camadas, LR baixa
        base.trainable = True
        for camada in base.layers[:-30]:
            camada.trainable = False
        modelo.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss='binary_crossentropy', metrics=['accuracy'])
        h2 = modelo.fit(ger_treino, validation_data=ger_val, epochs=15, callbacks=[callback()])
    return modelo, h1, h2, (time.time() - t0) / 60

def avalia(modelo, ger_teste):
    """Acurácia no teste e tempo de inferência por imagem (uma por vez, como o detect.py)."""
    ger_teste.reset()
    previsto = (modelo.predict(ger_teste, verbose=0).ravel() > 0.5).astype(int)
    verdadeiro = ger_teste.classes
    acertos = int((previsto == verdadeiro).sum())
    uma = np.expand_dims(next(iter(ger_teste))[0][0], 0)
    modelo.predict(uma, verbose=0)
    t0 = time.time()
    for _ in range(20):
        modelo.predict(uma, verbose=0)
    return acertos / len(verdadeiro), acertos, len(verdadeiro), (time.time() - t0) / 20 * 1000, verdadeiro, previsto

print('tensorflow', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU') != [])

## 5. Quatro treinos: (original × recortada) × (do zero × pré-treinada)

Assim as duas hipóteses são testadas nas mesmas condições — mesmo split, mesma entrada, mesmas épocas e mesma augmentação. Só muda a base (com ou sem o fundo recortado) e a arquitetura (do zero ou pré-treinada).

In [ ]:
# Treina as 4 combinações e guarda os resultados numa tabela
import pandas as pd

BASES = {'original': LOCAL, 'recortada': f'{PASTA_OBJ}'}
ARQUITETURAS = {'CNN do zero': cnn_do_zero, 'MobileNetV2 (ImageNet)': mobilenet}

resultados, treinados = [], {}
for nome_base, raiz in BASES.items():
    gt, gv, gte = geradores(raiz)
    for nome_arq, construtor in ARQUITETURAS.items():
        modelo, h1, h2, minutos = treina(construtor, gt, gv)
        acuracia, acertos, total, ms, verdadeiro, previsto = avalia(modelo, gte)
        resultados.append({
            'base': nome_base, 'arquitetura': nome_arq,
            'acurácia (teste)': round(acuracia, 3),
            'acertos': f'{acertos}/{total}',
            'épocas': len(h1.history['loss']) + (len(h2.history['loss']) if h2 else 0),
            'tempo de treino (min)': round(minutos, 2),
            'inferência (ms/imagem)': round(ms, 1),
            'parâmetros': modelo.count_params(),
        })
        treinados[(nome_base, nome_arq)] = (modelo, h1, h2, gte)
        print(f'{nome_base} + {nome_arq}: acurácia {acuracia:.3f} ({acertos}/{total}), {minutos:.2f} min')

tabela = pd.DataFrame(resultados)
display(tabela)

### Curvas de treino (treino × validação)

A perda de validação subindo enquanto a de treino continua caindo é o retrato do overfitting — o ponto em que a rede começa a decorar as 64 imagens de treino.

In [ ]:
# Curvas dos 4 treinos (as duas fases do MobileNetV2 aparecem emendadas)
import matplotlib.pyplot as plt

cores = {('original', 'CNN do zero'): 'tab:blue', ('original', 'MobileNetV2 (ImageNet)'): 'tab:orange',
         ('recortada', 'CNN do zero'): 'tab:green', ('recortada', 'MobileNetV2 (ImageNet)'): 'tab:red'}
fig, eixos = plt.subplots(1, 2, figsize=(12, 4))
for (base_nome, arq), (_, h1, h2, _) in treinados.items():
    perda_treino = h1.history['loss'] + (h2.history['loss'] if h2 else [])
    perda_val = h1.history['val_loss'] + (h2.history['val_loss'] if h2 else [])
    eixos[0].plot(perda_treino, color=cores[(base_nome, arq)], label=f'{base_nome} + {arq}')
    eixos[0].plot(perda_val, color=cores[(base_nome, arq)], linestyle='--')
    acc = h1.history['accuracy'] + (h2.history['accuracy'] if h2 else [])
    acc_val = h1.history['val_accuracy'] + (h2.history['val_accuracy'] if h2 else [])
    eixos[1].plot(acc, color=cores[(base_nome, arq)], label=f'{base_nome} + {arq}')
    eixos[1].plot(acc_val, color=cores[(base_nome, arq)], linestyle='--')
for eixo, titulo in zip(eixos, ('perda', 'acurácia')):
    eixo.set_title(f'{titulo} — linha cheia: treino, tracejada: validação'); eixo.set_xlabel('época'); eixo.legend(fontsize=8)
plt.tight_layout(); plt.show()

## 6. Matriz de confusão dos quatro modelos

Com 8 imagens de teste (4 de cada classe), cada erro vale 12,5 pontos de acurácia — a matriz mostra *onde* cada modelo erra, o que a acurácia sozinha esconde.

In [ ]:
# Uma matriz de confusão por combinação
fig, eixos = plt.subplots(1, 4, figsize=(16, 4))
for eixo, ((nome_base, nome_arq), (modelo, h1, h2, gte)) in zip(eixos, treinados.items()):
    gte.reset()
    previsto = (modelo.predict(gte, verbose=0).ravel() > 0.5).astype(int)
    ConfusionMatrixDisplay(confusion_matrix(gte.classes, previsto),
                           display_labels=list(gte.class_indices)).plot(ax=eixo, cmap='Blues', colorbar=False)
    eixo.set_title(f'{nome_base} + {nome_arq}', fontsize=9)
plt.tight_layout(); plt.show()

## 7. Arquitetura da solução

Figura autoral do pipeline: a base da Entrega 1 entra na segmentação automática (COCO), a máscara gera a versão recortada, e as duas versões alimentam os dois classificadores — é exatamente o desenho das duas hipóteses testadas.

In [ ]:
# Figura de arquitetura, desenhada em código (matplotlib)
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

fig, ax = plt.subplots(figsize=(11, 4.6))
ax.axis('off')

def caixa(x, y, w, h, texto, cor):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.02', facecolor=cor, edgecolor='black', linewidth=1))
    ax.text(x + w / 2, y + h / 2, texto, ha='center', va='center', fontsize=9, wrap=True)

def seta(x1, y1, x2, y2, texto=''):
    ax.add_patch(FancyArrowPatch((x1, y1), (x2, y2), arrowstyle='-|>', mutation_scale=12, linewidth=1.2))
    if texto:
        ax.text((x1 + x2) / 2, (y1 + y2) / 2 + 0.04, texto, ha='center', fontsize=8, style='italic')

caixa(0.02, 0.55, 0.17, 0.28, 'base da Entrega 1\n80 imagens (vaca / caminhão)\nsplit 32/4/4', '#dce9ff')
caixa(0.24, 0.55, 0.18, 0.28, 'segmentação automática\nYOLO-seg (COCO)\nclasses cow(19) / truck(7)', '#ffe9cc')
caixa(0.47, 0.78, 0.16, 0.20, 'máscara\n(união dos objetos\nda classe)', '#e6ffe6')
caixa(0.47, 0.32, 0.16, 0.20, 'imagem recortada\nfundo preto', '#e6ffe6')
caixa(0.70, 0.72, 0.26, 0.22, 'CNN do zero\n(3× Conv2D+MaxPool\n+ densas)', '#f0e6ff')
caixa(0.70, 0.40, 0.26, 0.22, 'MobileNetV2\nImageNet + fine tuning\n(últimas 30 camadas)', '#f0e6ff')
caixa(0.70, 0.06, 0.26, 0.20, 'comparação das 4 combinações\n(acurácia, tempo, ms/imagem)', '#ffd9d9')
seta(0.19, 0.69, 0.24, 0.69)
seta(0.42, 0.72, 0.47, 0.86); seta(0.42, 0.64, 0.47, 0.44)
seta(0.63, 0.88, 0.70, 0.83); seta(0.63, 0.42, 0.70, 0.51)
seta(0.83, 0.72, 0.83, 0.62); seta(0.83, 0.40, 0.83, 0.26)
ax.text(0.5, 0.02, 'hipótese 1: pré-treinada > do zero?    hipótese 2: recortar o objeto ajuda?',
        ha='center', fontsize=9, style='italic')
plt.tight_layout()
plt.savefig('/content/arquitetura_ir_alem.png', dpi=130)
plt.show()
print('figura salva em /content/arquitetura_ir_alem.png')

## 8. Guardar as figuras no Drive

As duas figuras (exemplos de segmentação e arquitetura) ficam junto dos resultados, para usar no vídeo e no PDF da entrega.

In [ ]:
# Copia as figuras e a tabela para a pasta de resultados no Drive
SAIDA = '/content/drive/MyDrive/Fase 6 - Cap 1/ir_alem'
os.makedirs(SAIDA, exist_ok=True)
for arq in ('/content/segmentacao_exemplos.png', '/content/arquitetura_ir_alem.png'):
    shutil.copy(arq, SAIDA)
tabela.to_csv(f'{SAIDA}/comparacao_ir_alem.csv', index=False)
print('salvo em', SAIDA, '->', sorted(os.listdir(SAIDA)))

## 9. Leitura dos resultados

_(Esta seção é preenchida com os números da execução — a tabela, as curvas e as matrizes acima são a evidência.)_

**Hipótese 1 — Transfer Learning vale a pena com 64 imagens?** Compare as linhas de mesma base: **MobileNetV2 (ImageNet)** contra **CNN do zero**. O que olhar: acurácia no teste, épocas necessárias e tempo de treino.

**Hipótese 2 — Pré-segmentar ajuda?** Compare as linhas de mesma arquitetura: base **recortada** contra **original**. Se recortar ajuda, a acurácia sobe com o mesmo número de imagens — sinal de que o fundo estava atrapalhando a decisão.

**Limitações:** a segmentação não é perfeita (as imagens em que o YOLO-seg não achou o objeto entram com a imagem original, e isso está registrado acima); a máscara pode cortar parte do animal; e o teste tem 8 imagens, então 1 erro = 12,5 pontos de acurácia.